# ORC Basics — 04: Predicate Pushdown

ORC's predicate pushdown uses 3 levels of statistics.
Bloom filters add a fourth layer for equality predicates.

Topics: row-level stats, bloom filter pushdown, supported predicates, explain().


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:23:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Setup: Sorted ORC for Best Statistics



In [2]:

ORC_PATH = f"{DATA_DIR}/pushdown_test.orc"
import random, datetime
random.seed(42); N=300_000
rows=[]
base=datetime.date(2023,1,1)
CATS=["Electronics","Clothing","Books","Food","Sports","Furniture"]
CTRS=["US","UK","DE","FR","JP","CA"]
for i in range(N):
    d=base+datetime.timedelta(days=random.randint(0,365))
    rows.append((i+1,random.randint(1,10000),random.choice(CATS),random.choice(CTRS),round(random.uniform(5,1000),2),d))
df_src = spark.createDataFrame(rows,["order_id","customer_id","category","country","revenue","order_date"])
df_src.orderBy("category","country","revenue") \
      .write.mode("overwrite") \
      .option("orc.bloom.filter.columns","category,country,status") \
      .orc(ORC_PATH)
print(f"Written {N:,} rows sorted → good statistics")


26/04/10 13:23:22 WARN TaskSetManager: Stage 0 contains a task of very large size (1865 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 13:23:24 WARN TaskSetManager: Stage 1 contains a task of very large size (1865 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Written 300,000 rows sorted → good statistics


## Step 2 — Verify Pushdown with explain()



In [3]:

# Equality filter — bloom filter + min/max
df_eq = spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics")
print("Equality filter plan:")
df_eq.explain(mode="formatted")
print("Look for PushedFilters in OrcScan")


Equality filter plan:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan orc  (1)


(1) Scan orc 
Output [6]: [order_id#6L, customer_id#7L, category#8, country#9, revenue#10, order_date#11]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/orc_basics/pushdown_test.orc]
PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,category:string,country:string,revenue:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [order_id#6L, customer_id#7L, category#8, country#9, revenue#10, order_date#11]

(3) Filter [codegen id : 1]
Input [6]: [order_id#6L, customer_id#7L, category#8, country#9, revenue#10, order_date#11]
Condition : (isnotnull(category#8) AND (category#8 = Electronics))


Look for PushedFilters in OrcScan


## Step 3 — Range Filter (min/max statistics)



In [4]:

# Range filter uses min/max stats to skip stripes
df_range = spark.read.orc(ORC_PATH).filter(F.col("revenue").between(800, 1000))
print("Range filter plan:")
df_range.explain(mode="formatted")

runs=3
tf,tr=[],[]
for _ in range(runs):
    t0=time.time(); spark.read.orc(ORC_PATH).agg(F.sum("revenue")).collect(); tf.append(time.time()-t0)
    t0=time.time(); spark.read.orc(ORC_PATH).filter(F.col("revenue")>900).agg(F.sum("revenue")).collect(); tr.append(time.time()-t0)
print(f"\nFull scan         : {sorted(tf)[1]:.3f}s")
print(f"revenue > 900 scan: {sorted(tr)[1]:.3f}s  (stripe/row-index skipping)")


Range filter plan:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan orc  (1)


(1) Scan orc 
Output [6]: [order_id#13L, customer_id#14L, category#15, country#16, revenue#17, order_date#18]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/orc_basics/pushdown_test.orc]
PushedFilters: [IsNotNull(revenue), GreaterThanOrEqual(revenue,800.0), LessThanOrEqual(revenue,1000.0)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,category:string,country:string,revenue:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [order_id#13L, customer_id#14L, category#15, country#16, revenue#17, order_date#18]

(3) Filter [codegen id : 1]
Input [6]: [order_id#13L, customer_id#14L, category#15, country#16, revenue#17, order_date#18]
Condition : ((isnotnull(revenue#17) AND (revenue#17 >= 800.0)) AND (revenue#17 <= 1000.0))





Full scan         : 0.372s
revenue > 900 scan: 0.402s  (stripe/row-index skipping)


## Step 4 — What Does NOT Push Down



In [5]:

tests = [
    ("EqualTo",    "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics")),
    ("GreaterThan","✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("revenue")>500)),
    ("In",         "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("category").isin("Electronics","Books"))),
    ("IsNull",     "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("revenue").isNull())),
    ("Contains",   "❌", lambda: spark.read.orc(ORC_PATH).filter(F.col("category").contains("Elect"))),
    ("UDF",        "❌", lambda: spark.read.orc(ORC_PATH).filter(F.udf(lambda x: x=="Electronics","boolean")(F.col("category")))),
]
print(f"{'Predicate':<15} {'Pushed?':>8}")
print("-"*28)
for label,expected,fn in tests:
    plan=fn()._jdf.queryExecution().executedPlan().toString()
    pushed="✅ YES" if "PushedFilters" in plan and "[]" not in plan.split("PushedFilters")[1][:50] else "❌ NO"
    print(f"  {label:<13} {pushed:>8}  (expected {expected})")


Predicate        Pushed?
----------------------------
  EqualTo          ✅ YES  (expected ✅)
  GreaterThan      ✅ YES  (expected ✅)
  In               ✅ YES  (expected ✅)
  IsNull           ✅ YES  (expected ✅)
  Contains         ✅ YES  (expected ❌)
  UDF               ❌ NO  (expected ❌)


## Summary



In [6]:

print("""
ORC predicate pushdown:
  ✅ EqualTo, LessThan, GreaterThan, LessThanOrEqual, GreaterThanOrEqual
  ✅ In (equality list)
  ✅ IsNull, IsNotNull
  ✅ And, Or combinations
  ❌ Contains / LIKE '%...'
  ❌ Python UDFs
  ❌ Derived/computed columns

Bloom filters add a FOURTH level:
  File stats → Stripe stats → Row index stats → Bloom filter → Row data
  Best for: equality filters on high-cardinality string/int columns
""")



ORC predicate pushdown:
  ✅ EqualTo, LessThan, GreaterThan, LessThanOrEqual, GreaterThanOrEqual
  ✅ In (equality list)
  ✅ IsNull, IsNotNull
  ✅ And, Or combinations
  ❌ Contains / LIKE '%...'
  ❌ Python UDFs
  ❌ Derived/computed columns

Bloom filters add a FOURTH level:
  File stats → Stripe stats → Row index stats → Bloom filter → Row data
  Best for: equality filters on high-cardinality string/int columns



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 56304)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin